# Economy of Bits — Notebook 02: Panel repair

Notebook 01 found that the JRFM panel **loses firms when their ticker changes**. Yahoo is
keyed on current tickers, so a historical ticker returns nothing and the firm enters the
panel only in the year it was renamed. Meta is absent from 2016–2021 and present from 2022,
having been in the index continuously since 2013. Fiserv runs the other way — present
through 2022, gone from 2023, the year it renamed FISV → FI.

Roster coverage climbs 80.8% → 97.0% across exactly the window every trend is estimated
over. Part of that is genuine survivorship. This notebook measures how much of it is not.

**One job: repair the panel, re-run the paper's own estimators on it, and report how far
the published numbers move.** The bit-intensity index waits. A defect in a paper under
review takes precedence over a measurement in a paper not yet written.

Upload `bits_inputs_v2.zip` to the file pane first. Runtime is about 15 minutes — only the
recovered firms are downloaded, not the whole index.

## 1 · Setup

In [ ]:
# =============================================================================
# CELL 1 - Setup
#
# Notebook 02 has one job: REPAIR THE PANEL, then re-run the JRFM estimates on
# the repaired version and report how far the published numbers move.
#
# Notebook 01 found that the panel loses firms when their TICKER changes. Yahoo
# is keyed on current tickers, so a historical ticker returns nothing and the
# firm enters the panel only in the year it was renamed. Meta is absent from
# 2016-2021 and present from 2022, having been in the index continuously since
# 2013. Fiserv runs the other way: present through 2022, gone from 2023, the
# year it renamed FISV -> FI, while staying in the index throughout.
#
# The bit-intensity index waits until this is settled. A defect in a paper under
# review takes precedence over a new measurement in a paper not yet written.
# =============================================================================
import os, re, sys, json, time, glob, zipfile, pathlib, datetime, shutil, warnings
import numpy as np, pandas as pd, requests

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

try:
    import yfinance as yf
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=True)
    import yfinance as yf

RUN_TAG = "bits_02_panel_repair"
BASE = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
ROOT = BASE / f"economy_of_bits_{RUN_TAG}"
OUTD, CACHE = ROOT / "outputs", ROOT / "cache"
for d in (OUTD, CACHE):
    d.mkdir(parents=True, exist_ok=True)
saved_tables = []

def savetable(df, name):
    if df is None or not len(df):
        print(f"  skipped {name} (empty)"); return
    df.to_parquet(OUTD / f"{name}.parquet")
    df.to_csv(OUTD / f"{name}.csv", index=False)
    saved_tables.append(name)
    print(f"  saved {name}  ({len(df):,} rows)")

def cached(name, builder, refresh=False):
    p = CACHE / f"{name}.parquet"
    if p.exists() and not refresh:
        d = pd.read_parquet(p); print(f"  cache hit: {name} ({len(d):,} rows)"); return d
    d = builder()
    if d is not None and len(d):
        d.to_parquet(p); print(f"  cached {name} ({len(d):,} rows)")
    return d

SEC_UA = "Boston University Metropolitan College research (spattna1@bu.edu)"
SEC_HEADERS = {"User-Agent": SEC_UA, "Accept-Encoding": "gzip, deflate"}
SEC_SLEEP = 0.15

def sec_get(url, tries=3, timeout=60):
    for attempt in range(tries):
        try:
            time.sleep(SEC_SLEEP)
            r = requests.get(url, headers=SEC_HEADERS, timeout=timeout)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 404:
                return None
            if r.status_code in (403, 429):
                time.sleep(5 * (attempt + 1)); continue
            return None
        except Exception:
            if attempt == tries - 1:
                return None
            time.sleep(2 * (attempt + 1))
    return None

PAPER_YEARS = list(range(2016, 2026))
print(f"writing to: {ROOT}")

## 2 · Inputs, and the defect restated

In [ ]:
# =============================================================================
# CELL 2 - Inputs
#
# Upload bits_inputs_v2.zip to the Colab file pane. It carries the JRFM panel,
# the reconstructed rosters, the DATED membership snapshots (needed in cell 3),
# and the published result tables this notebook has to reproduce and then
# compare against.
# =============================================================================
def find_inputs():
    for z in sorted(glob.glob(str(BASE / "**" / "bits_inputs_v2.zip"), recursive=True)):
        dest = CACHE / "inputs"; dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        print(f"  extracted {pathlib.Path(z).name}")
        return dest
    for d in (BASE / "inputs2", BASE, CACHE / "inputs"):
        if (pathlib.Path(d) / "sp500_marketcap_panel_clean.csv.gz").exists():
            return pathlib.Path(d)
    return None

SRC = find_inputs()
if SRC is None:
    raise FileNotFoundError("bits_inputs_v2.zip not found. Upload it to the file pane.")

panel = pd.read_csv(SRC / "sp500_marketcap_panel_clean.csv.gz")
panel["year"] = panel.year.astype(int)
rosters_long = pd.read_csv(SRC / "sp500_membership_reconstructed.csv")
snapshots = pd.read_csv(SRC / "sp500_historical_components.csv.gz")
pub_ineq = pd.read_csv(SRC / "inequality_by_year.csv")
pub_alpha = pd.read_csv(SRC / "pareto_alpha_by_year.csv")
pub_group = pd.read_csv(SRC / "pareto_alpha_by_group.csv")
avail = pd.read_csv(SRC / "price_availability.csv")

est = panel[panel.year.isin(PAPER_YEARS)].copy()
ROSTER = {int(y): set(g.ticker) for y, g in rosters_long.groupby("year")
          if int(y) in PAPER_YEARS}

print(f"\npanel        {len(est):,} firm-years, {est.ticker.nunique()} tickers")
print(f"rosters      {len(ROSTER)} years, "
      f"{int(np.mean([len(v) for v in ROSTER.values()]))} members on average")
print(f"snapshots    {len(snapshots):,} dated constituent lists, "
      f"{snapshots.date.min()} to {snapshots.date.max()}")

cov = []
for y in PAPER_YEARS:
    got = set(est[est.year == y].ticker)
    cov.append({"year": y, "roster": len(ROSTER[y]), "in_panel": len(got),
                "missing": len(ROSTER[y] - got),
                "coverage": len(got) / len(ROSTER[y])})
coverage_before = pd.DataFrame(cov)
print("\nthe defect, restated:")
print(coverage_before.assign(pct=(100*coverage_before.coverage).round(1))
      [["year", "roster", "in_panel", "missing", "pct"]].to_string(index=False))
print("\nCoverage climbs 17 points across the window. The paper reads that as")
print("survivorship. Cell 3 tests how much of it is renaming instead.")

## 3 · Detect renames from the membership file

A rename is invisible to a price API but visible in the constituent list: on the day it
happens the old ticker leaves and the new one arrives in the same snapshot. Detected from
the data rather than typed from memory — a typed list contains only the cases I thought of.

In [ ]:
# =============================================================================
# CELL 3 - Detect renames from the membership file itself
#
# A rename is invisible to a price API but perfectly visible in the index
# constituent list: on the day it happens, the old ticker leaves and the new one
# arrives IN THE SAME SNAPSHOT. Ordinary index changes also swap same-day, so
# this produces candidates, not conclusions. Cell 4 corroborates each one
# against EDGAR before anything is applied.
#
# Detecting renames from the data rather than typing a list matters: a typed
# list contains the cases I happened to think of, and the ones I did not think
# of are exactly the ones that would stay broken.
# =============================================================================
snap = snapshots.copy()
snap["date"] = pd.to_datetime(snap.date)
snap = snap.sort_values("date")
snap["members"] = snap.tickers.astype(str).str.split(",").apply(
    lambda xs: {x.strip().upper() for x in xs if x.strip()})

prev_date, prev = None, None
events = []
for _, row in snap.iterrows():
    cur = row.members
    if prev is not None:
        added, dropped = cur - prev, prev - cur
        if added and dropped:
            events.append({"date": row.date, "prev_date": prev_date,
                           "added": sorted(added), "dropped": sorted(dropped)})
    prev, prev_date = cur, row.date

print(f"{len(events)} snapshot transitions add and drop on the same day")

# A rename is a ONE-FOR-ONE swap: exactly one out, exactly one in. A day on
# which three firms enter and two leave is index rebalancing, not a rename.
one_for_one = [e for e in events if len(e["added"]) == 1 and len(e["dropped"]) == 1]
print(f"{len(one_for_one)} of those are one-for-one swaps -- the rename shape")

cands = pd.DataFrame([{"date": e["date"], "old": e["dropped"][0],
                       "new": e["added"][0]} for e in one_for_one])
cands = cands[cands.date.dt.year.between(min(PAPER_YEARS) - 1, max(PAPER_YEARS))]

# A rename keeps the firm in the index, so the new ticker must persist. A
# genuine index substitution usually does too, which is why cell 4 exists.
last_members = snap.iloc[-1].members
cands["new_still_in_index"] = cands.new.isin(last_members)

# Ticker changes very often preserve a prefix (FISV->FI, BLL->BALL, ANTM->ELV
# does not). Prefix agreement is weak evidence, recorded but never decisive.
def shared_prefix(a, b):
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n
cands["shared_prefix"] = [shared_prefix(o, n) for o, n in zip(cands.old, cands.new)]

# Did the old ticker have usable prices? If it did, the firm is already in the
# panel and there is nothing to recover.
has_px = dict(zip(avail.ticker, avail.has_prices.astype(bool)))
cands["old_had_prices"] = cands.old.map(has_px).fillna(False)
cands = cands.sort_values("date").reset_index(drop=True)

print(f"\n{len(cands)} one-for-one swaps inside the estimation window:")
display(cands.assign(date=cands.date.dt.date))
print("\nMost of these will be ordinary index substitutions. Cell 4 asks EDGAR")
print("which ones are the same company under a new name.")
savetable(cands.assign(date=cands.date.astype(str)), "rename_candidates_raw")

## 4 · Corroborate each candidate against EDGAR

One company means one CIK, one continuous filing history spanning both tickers' years, and
a `formerNames` entry. Two companies mean an ordinary index substitution.

In [ ]:
# =============================================================================
# CELL 4 - Corroborate each candidate against EDGAR
#
# The test that separates a rename from an index substitution:
#
#   A RENAME means one company, one CIK, one continuous filing history that
#   spans BOTH the old ticker's years and the new ticker's years, and a
#   `formerNames` entry in EDGAR's submissions record.
#
#   A SUBSTITUTION means two companies and two CIKs, and the incoming firm's
#   filing history has no reason to reach back before it joined.
#
# EDGAR's submissions endpoint carries `formerNames` with dates, which is the
# closest thing to an authoritative record of corporate renaming that exists in
# public data.
# =============================================================================
def fetch_cik_map():
    r = requests.get("https://www.sec.gov/files/company_tickers.json",
                     headers=SEC_HEADERS, timeout=60)
    r.raise_for_status()
    d = pd.DataFrame(r.json()).T
    d.columns = [c.lower() for c in d.columns]
    d = d.rename(columns={"cik_str": "cik", "title": "sec_name"})
    d["ticker"] = d.ticker.str.upper().str.strip()
    d["cik"] = d.cik.astype(int)
    return d[["ticker", "cik", "sec_name"]]

cik_map = cached("sec_cik_map", fetch_cik_map)
T2C = dict(zip(cik_map.ticker, cik_map.cik))

def submissions(cik):
    return sec_get(f"https://data.sec.gov/submissions/CIK{int(cik):010d}.json")

def filing_span(js):
    """First and last 10-K filing date on record, from the recent filings block
    plus any older paged files."""
    if not js:
        return None, None
    dates = []
    rec = js.get("filings", {}).get("recent", {})
    forms, filed = rec.get("form", []), rec.get("filingDate", [])
    dates += [d for f, d in zip(forms, filed) if f.startswith("10-K")]
    for extra in js.get("filings", {}).get("files", []):
        sub = sec_get(f"https://data.sec.gov/submissions/{extra['name']}")
        if sub:
            dates += [d for f, d in zip(sub.get("form", []), sub.get("filingDate", []))
                      if f.startswith("10-K")]
    if not dates:
        return None, None
    return min(dates), max(dates)

def check_candidates(c):
    rows = []
    for i, row in c.iterrows():
        new_cik = T2C.get(row.new)
        old_cik = T2C.get(row.old)
        rec = {"date": row.date, "old": row.old, "new": row.new,
               "new_cik": new_cik, "old_cik_current": old_cik,
               "old_had_prices": bool(row.old_had_prices),
               "former_names": "", "first_10k": None, "last_10k": None,
               "spans_old_years": False, "verdict": "unresolved"}
        if new_cik is None:
            rec["verdict"] = "new ticker not in SEC map"
            rows.append(rec); continue
        js = submissions(new_cik)
        if js:
            fn = js.get("formerNames", []) or []
            rec["former_names"] = "; ".join(
                f"{f.get('name')} (to {str(f.get('to', ''))[:10]})" for f in fn)[:180]
            rec["entity"] = js.get("name")
            f10, l10 = filing_span(js)
            rec["first_10k"], rec["last_10k"] = f10, l10
            if f10:
                # Does its own filing history reach back before the swap? A
                # renamed firm's does by construction; a firm newly added to the
                # index may or may not, which is why formerNames is the stronger
                # signal and this is corroboration.
                rec["spans_old_years"] = pd.Timestamp(f10) < row.date - pd.Timedelta(days=400)

        # The decisive combination. Only two verdicts are strong enough to
        # apply automatically.
        #
        # An earlier version of this rule also accepted "filing history predates
        # the swap, and the old ticker is not in the current SEC map". A dry run
        # showed why that is too weak: it paired INFO -> MOH, because IHS Markit
        # left the index the day Molina joined and Molina -- an established
        # filer -- of course has filings predating 2022. Applying that alias
        # would have attributed Molina's market capitalisation to IHS Markit's
        # roster slot and counted Molina twice. EVERY established firm added to
        # the index passes that test, so it carries no information.
        if old_cik is not None and new_cik is not None and old_cik == new_cik:
            rec["verdict"] = "RENAME (same CIK)"
        elif rec["former_names"] and rec["spans_old_years"] and old_cik is None:
            rec["verdict"] = "RENAME (formerNames + history predates swap)"
        elif rec["spans_old_years"] and old_cik is None:
            rec["verdict"] = "REVIEW (history predates swap, no formerNames)"
        else:
            rec["verdict"] = "index substitution"
        rows.append(rec)
        if (i + 1) % 10 == 0:
            print(f"    {i+1}/{len(c)} checked")
    return pd.DataFrame(rows)

checked = cached("rename_checked", lambda: check_candidates(cands))
checked["date"] = pd.to_datetime(checked.date)
print("\nverdicts:")
print(checked.verdict.value_counts().to_string())
print("\nfull table -- READ THIS BEFORE RUNNING CELL 5:")
display(checked.assign(date=checked.date.dt.date)
        [["date", "old", "new", "entity", "new_cik", "old_cik_current",
          "old_had_prices", "first_10k", "spans_old_years", "verdict"]]
        if "entity" in checked.columns else checked)
savetable(checked.assign(date=checked.date.astype(str)), "rename_verdicts")

## 5 · The alias map, reviewed rather than auto-applied

The detector's verdicts are compared against the list I built by hand in notebook 01. Where
they disagree, the detector wins — a check that only ever agrees with me is worth little.

In [ ]:
# =============================================================================
# CELL 5 - The alias map, reviewed rather than auto-applied
#
# Detection produces candidates; a human decides. The seed below is the set I
# identified from notebook 01 by hand and is stated explicitly so that it can be
# checked against cell 4's independent verdicts. Where the two disagree, cell 4
# wins and the disagreement is printed -- an automated check that agrees with me
# is worth little, and one that contradicts me is worth a great deal.
# =============================================================================
SEED_ALIASES = {
    "FB":    "META",   # Meta Platforms, renamed June 2022
    "ANTM":  "ELV",    # Elevance Health, renamed June 2022
    "DISCA": "WBD",    # Warner Bros. Discovery, April 2022
    "BLL":   "BALL",   # Ball Corporation, 2022
    "ABC":   "COR",    # Cencora, August 2023
    "RE":    "EG",     # Everest Group, 2023
    "PKI":   "RVTY",   # Revvity, May 2023
    "PEAK":  "DOC",    # Healthpeak Properties, 2024
    "STI":   "TFC",    # Truist, after the BB&T/SunTrust merger, Dec 2019
    "FISV":  "FI",     # Fiserv, 2023 -- the case that runs the other way
    "COG":   "CTRA",   # Coterra Energy, Oct 2021
    "INFO":  "SPGI",   # IHS Markit into S&P Global, Feb 2022 (merger, not rename)
    "CBS":   "PARA",   # ViacomCBS -> Paramount
}

# Only the two strong verdicts are applied. "REVIEW" rows are printed for the
# next iteration and deliberately NOT used.
auto = {r.old: r.new for _, r in checked.iterrows()
        if str(r.verdict).startswith("RENAME")}
review = checked[checked.verdict.str.startswith("REVIEW")]
if len(review):
    print(f"{len(review)} candidates need human review and are NOT applied:")
    display(review[["date", "old", "new", "entity", "first_10k"]]
            if "entity" in review.columns else review[["date", "old", "new"]])
    print()
print(f"cell 4 found {len(auto)} renames automatically: {auto}")
print(f"my seed list has {len(SEED_ALIASES)}")

only_auto = {k: v for k, v in auto.items() if k not in SEED_ALIASES}
only_seed = {k: v for k, v in SEED_ALIASES.items() if k not in auto}
disagree = {k: (SEED_ALIASES[k], auto[k]) for k in set(auto) & set(SEED_ALIASES)
            if SEED_ALIASES[k] != auto[k]}
print(f"\nfound by the detector but NOT on my list ({len(only_auto)}): {only_auto}")
print("  -> these are the valuable ones. I did not think of them.")
print(f"\non my list but NOT found by the detector ({len(only_seed)}): {only_seed}")
print("  -> either the swap was not one-for-one on a single snapshot date, or")
print("     the old ticker never appears in the window. Kept anyway ONLY if the")
print("     two tickers resolve to the same CIK; checked below.")
if disagree:
    print(f"\nDISAGREEMENT on the target ticker: {disagree}")
    print("  -> the detector wins. Investigate each before proceeding.")

# Verify every seed entry the detector missed, by CIK. This is the check that
# stops a plausible-looking but wrong alias entering the panel.
verified, rejected = {}, {}
for old, new in SEED_ALIASES.items():
    if old in auto:
        verified[old] = auto[old]; continue
    ncik = T2C.get(new)
    if ncik is None:
        rejected[old] = f"{new} is not in the SEC ticker map"
        continue
    js = submissions(ncik)
    fn = "; ".join(f.get("name", "") for f in (js or {}).get("formerNames", []) or [])
    f10, _ = filing_span(js)
    old_years = sorted(y for y in PAPER_YEARS if old in ROSTER.get(y, set()))
    if f10 and old_years and pd.Timestamp(f10).year <= min(old_years):
        verified[old] = new
        print(f"  verified {old} -> {new}: {(js or {}).get('name')}, "
              f"10-Ks from {f10}, former names: {fn[:70] or 'none recorded'}")
    else:
        rejected[old] = (f"{new} (CIK {ncik}) first 10-K {f10}, "
                         f"old ticker in roster {old_years[:3] if old_years else 'never'}")

ALIASES = {**auto, **verified}
print(f"\nFINAL ALIAS MAP: {len(ALIASES)} entries")
for k, v in sorted(ALIASES.items()):
    print(f"  {k:<6} -> {v}")
if rejected:
    print(f"\nREJECTED ({len(rejected)}), not applied:")
    for k, v in rejected.items():
        print(f"  {k}: {v}")

# --- final safety check: an alias cannot join two firms that coexisted -------
# If the old and new tickers are both in the index in the same year they are two
# different companies, whatever the other evidence says.
collisions = []
for old_t, new_t in list(ALIASES.items()):
    both = [y for y in PAPER_YEARS
            if old_t in ROSTER.get(y, set()) and new_t in ROSTER.get(y, set())]
    if both:
        collisions.append((old_t, new_t, both))
        del ALIASES[old_t]
if collisions:
    print(f"\nREMOVED {len(collisions)} aliases -- both tickers are in the index")
    print("in the same year, so they cannot be one firm:")
    for o, n, ys in collisions:
        print(f"  {o} -> {n}: both present in {ys}")
else:
    print("\ncollision check: no alias joins two firms that coexisted in the index")

alias_df = pd.DataFrame([{"old": k, "new": v,
                          "source": "detector" if k in auto else "seed, CIK-verified"}
                         for k, v in sorted(ALIASES.items())])
savetable(alias_df, "ticker_aliases_applied")

## 6 · What is recoverable, and what is honestly lost

Renaming is recoverable and leaving it unrecovered is an error. Acquisition is not, and
that residual is the survivorship limitation the paper should report.

In [ ]:
# =============================================================================
# CELL 6 - What is now recoverable, and what is honestly lost
#
# Two channels have been conflated in the paper's coverage figure and they have
# different standing:
#
#   RENAMING -- the firm is alive and in the index; only the label changed.
#   Recoverable, and leaving it unrecovered is an error.
#
#   ACQUISITION or DELISTING -- the firm stopped trading. Yahoo serves no
#   history for it. Not recoverable from this source, and the residual is the
#   honest survivorship limitation the paper should report.
#
# Separating them is the point of this cell.
# =============================================================================
in_panel = {y: set(est[est.year == y].ticker) for y in PAPER_YEARS}
rows = []
for y in PAPER_YEARS:
    for t in sorted(ROSTER[y] - in_panel[y]):
        target = ALIASES.get(t)
        rows.append({"year": y, "roster_ticker": t,
                     "current_ticker": target or t,
                     "channel": "rename" if target else "no alias",
                     "in_sec_map": (target or t) in T2C})
gaps = pd.DataFrame(rows)

# A firm with no alias but a live SEC ticker is still trading under its own
# name, so its absence is a Yahoo failure rather than a corporate event. Marsh
# & McLennan is the known example: continuously listed, never renamed, and no
# price history at all in the first paper's download.
gaps.loc[(gaps.channel == "no alias") & gaps.in_sec_map, "channel"] = "still listed, Yahoo failed"
gaps.loc[(gaps.channel == "no alias") & ~gaps.in_sec_map, "channel"] = "acquired or delisted"

print("missing roster entries by channel and year:")
display(gaps.pivot_table(index="year", columns="channel", values="roster_ticker",
                         aggfunc="count").fillna(0).astype(int))

summary = (gaps.groupby("channel").roster_ticker.nunique()
           .rename("distinct_firms").to_frame())
summary["firm_years"] = gaps.groupby("channel").size()
print("\n")
display(summary)

RECOVER = sorted({r.current_ticker for _, r in gaps.iterrows()
                  if r.channel in ("rename", "still listed, Yahoo failed")})
print(f"\n{len(RECOVER)} tickers to download: {RECOVER}")
print("\nEverything else is genuine survivorship and stays missing. The paper")
print("reports it rather than repairing it.")
savetable(gaps, "coverage_gaps_by_channel")

## 7 · Download the recovered firms, on the same basis as the paper

In [ ]:
# =============================================================================
# CELL 7 - Download the recovered firms, on the SAME basis as the paper
#
# Every correction from the first paper has to be reproduced exactly, or the
# recovered firms sit on a different basis from the ones already in the panel
# and the comparison in cell 10 is meaningless.
#
#   M_it = P_it * S_it * prod(split ratios strictly after 31 Dec of year t)
#
# Yahoo's Close is split-adjusted to TODAY's basis; get_shares_full returns the
# ACTUAL contemporaneous count. Multiplying them understates every pre-split
# year by the cumulative split factor -- the largest error in the first paper,
# and the one most likely to be reintroduced here by accident.
# =============================================================================
def download_prices(tickers, batch=60):
    frames = []
    for i in range(0, len(tickers), batch):
        chunk = tickers[i:i + batch]
        try:
            px = yf.download(chunk, start="2014-01-01", end="2026-01-01",
                             progress=False, group_by="ticker", auto_adjust=False,
                             threads=True)
            for t in chunk:
                try:
                    s = px[t]["Close"] if isinstance(px.columns, pd.MultiIndex) else px["Close"]
                    s = s.dropna()
                    if len(s):
                        frames.append(s.rename(t))
                except Exception:
                    pass
        except Exception as e:
            print(f"    batch {i}: {type(e).__name__}")
        print(f"    {min(i+batch, len(tickers))}/{len(tickers)}")
    return pd.concat(frames, axis=1) if frames else pd.DataFrame()

def download_shares(tickers):
    rows = []
    for i, t in enumerate(tickers, 1):
        try:
            s = yf.Ticker(t).get_shares_full(start="2014-01-01")
            if s is not None and len(s):
                s = s[~s.index.duplicated(keep="last")]
                for dt, v in s.items():
                    rows.append({"ticker": t,
                                 "date": pd.Timestamp(dt).tz_localize(None),
                                 "shares": float(v)})
        except Exception:
            pass
        if i % 25 == 0:
            print(f"    {i}/{len(tickers)}")
        time.sleep(0.05)
    return pd.DataFrame(rows, columns=["ticker", "date", "shares"])

def download_splits(tickers):
    rows = []
    for t in tickers:
        try:
            s = yf.Ticker(t).splits
            if s is not None and len(s):
                for dt, ratio in s.items():
                    if ratio and float(ratio) > 0:
                        rows.append({"ticker": t,
                                     "date": pd.Timestamp(dt).tz_localize(None),
                                     "ratio": float(ratio)})
        except Exception:
            pass
        time.sleep(0.03)
    return pd.DataFrame(rows, columns=["ticker", "date", "ratio"])

print(f"downloading {len(RECOVER)} tickers")
px_new = cached("recovered_prices", lambda: download_prices(RECOVER))
sh_new = cached("recovered_shares", lambda: download_shares(RECOVER))
sp_new = cached("recovered_splits", lambda: download_splits(RECOVER))

got = [t for t in RECOVER if t in px_new.columns] if len(px_new) else []
print(f"\nprices for {len(got)} of {len(RECOVER)}")
failed = sorted(set(RECOVER) - set(got))
if failed:
    print(f"still no data for {len(failed)}: {failed}")
    print("  These stay out of the repaired panel and are reported as lost.")
print(f"share counts for {sh_new.ticker.nunique() if len(sh_new) else 0} tickers")
print(f"{len(sp_new)} split events")

## 8 · Market caps, with the split adjustment reproduced exactly

In [ ]:
# =============================================================================
# CELL 8 - Build market caps for the recovered firms
# =============================================================================
REV_ALIAS = {v: k for k, v in ALIASES.items()}

def year_end_prices(px):
    p = px.copy()
    p.index = pd.to_datetime(p.index)
    try:
        p.index = p.index.tz_localize(None)
    except (TypeError, AttributeError):
        p.index = p.index.tz_convert(None)
    return p.resample("YE").last()

def split_factor(t, year, sp):
    if not len(sp):
        return 1.0
    s = sp[(sp.ticker == t) & (sp.date > pd.Timestamp(f"{year}-12-31"))]
    return float(s.ratio.prod()) if len(s) else 1.0

ypx = year_end_prices(px_new) if len(px_new) else pd.DataFrame()
sh = sh_new.copy()
if len(sh):
    sh["year"] = pd.to_datetime(sh.date).dt.year
    ysh = sh.sort_values("date").groupby(["ticker", "year"]).shares.last().unstack(0)
    ysh = ysh.reindex(range(2014, 2027)).ffill().bfill()
else:
    ysh = pd.DataFrame()

recs = []
for y in PAPER_YEARS:
    rows = ypx.loc[ypx.index.year == y] if len(ypx) else pd.DataFrame()
    if not len(rows):
        continue
    prow = rows.iloc[-1]
    for roster_t in sorted(ROSTER[y] - in_panel[y]):
        cur = ALIASES.get(roster_t, roster_t)
        if cur not in prow.index or (len(ysh) and cur not in ysh.columns):
            continue
        p_ = prow.get(cur)
        s_ = ysh.at[y, cur] if (len(ysh) and y in ysh.index) else np.nan
        if pd.isna(p_) or pd.isna(s_) or p_ <= 0 or s_ <= 0:
            continue
        f = split_factor(cur, y, sp_new)
        recs.append({"year": y, "ticker": cur, "roster_ticker": roster_t,
                     "price": float(p_), "shares_adj": float(s_) * f,
                     "split_factor": f,
                     "mktcap_uncorrected": float(p_) * float(s_),
                     "mktcap": float(p_) * float(s_) * f,
                     "recovered": True})
recovered = pd.DataFrame(recs)
print(f"{len(recovered):,} recovered firm-years, "
      f"{recovered.ticker.nunique() if len(recovered) else 0} firms")
if len(recovered):
    n_split = int((recovered.split_factor != 1.0).sum())
    print(f"{n_split} of them carried a later split and were rescaled")
    print("\nby year:")
    print(recovered.groupby("year").size().to_string())

    # Validate against independently known values, exactly as the first paper
    # does. If Meta's 2021 market cap does not come back near $935bn, the
    # recovery is wrong and nothing below is worth reading.
    TRUTH_BN = {("META", 2021): 935, ("META", 2019): 585, ("META", 2017): 512,
                ("ELV", 2021): 113, ("WBD", 2022): 23, ("TFC", 2020): 64,
                ("CTRA", 2021): 15}
    chk = []
    for (t, y), true_bn in TRUTH_BN.items():
        d = recovered[(recovered.ticker == t) & (recovered.year == y)]
        if not len(d):
            continue
        got_bn = float(d.mktcap.iloc[0]) / 1e9
        chk.append({"ticker": t, "year": y, "recovered_bn": round(got_bn),
                    "reference_bn": true_bn, "ratio": round(got_bn / true_bn, 3)})
    check = pd.DataFrame(chk)
    if len(check):
        print("\nVALIDATION against independently known market caps:")
        display(check)
        worst = float((check.ratio - 1).abs().max())
        print(f"largest deviation {100*worst:.1f}%   "
              f"{'OK' if worst < 0.20 else 'STOP -- the recovery is wrong'}")
        savetable(check, "recovery_validation")
    savetable(recovered, "recovered_firm_years")

## 9 · Assemble and screen the repaired panel

In [ ]:
# =============================================================================
# CELL 9 - Assemble the repaired panel, and screen it the same way
#
# Same plausibility screen as the first paper: an absolute price ceiling and a
# share-of-index ceiling, applied only inside the estimation window. Deliberately
# narrow -- an earlier version also flagged prices far from a ticker's own
# median, which dropped NVIDIA, a legitimate firm whose split-adjusted price
# moved 13.5x.
# =============================================================================
PRICE_CEILING, MAX_FIRM_SHARE = 10_000.0, 0.15

base = est.copy(); base["recovered"] = False
base["roster_ticker"] = base.ticker
keep = ["year", "ticker", "roster_ticker", "price", "shares_adj", "mktcap",
        "sector", "sub_industry", "recovered"]
for c in keep:
    if c not in base.columns:
        base[c] = np.nan
    if len(recovered) and c not in recovered.columns:
        recovered[c] = np.nan

rep = pd.concat([base[keep], recovered[keep]], ignore_index=True) if len(recovered) \
      else base[keep].copy()

# One firm, one row per year. A recovered firm must not collide with itself:
# Meta arrives as META for 2016-2021 from recovery and is already META for
# 2022-2025 in the panel, so drop any duplicate (year, ticker) keeping the
# original.
before = len(rep)
rep = rep.sort_values(["year", "ticker", "recovered"]).drop_duplicates(
    ["year", "ticker"], keep="first")
print(f"{before - len(rep)} duplicate (year, ticker) rows collapsed")

# --- sector labels for the recovered rows ------------------------------------
# A recovered firm arrives with no sector, and cell 11 excludes unlabelled rows
# from the digital contrast. Without this step Meta would be recovered into the
# panel and then dropped from the one comparison the repair exists to fix.
#
# Three passes, cheapest first: carry the label within a ticker across years
# (Meta is labelled in its 2022-2025 rows), then within a firm across a rename
# (roster_ticker -> ticker), then ask Yahoo for whatever is still blank.
rep["sector"] = rep.sector.replace(["nan", "None", ""], np.nan)
known = (rep.dropna(subset=["sector"]).drop_duplicates("ticker")
         .set_index("ticker").sector)
rep["sector"] = rep.sector.fillna(rep.ticker.map(known))
rep["sector"] = rep.sector.fillna(rep.roster_ticker.map(known))

still = sorted(rep.loc[rep.sector.isna(), "ticker"].unique())
if still:
    print(f"\nasking Yahoo for sectors on {len(still)} tickers")
    # Yahoo's sector vocabulary is not GICS. Only the two labels the digital
    # split depends on are mapped; everything else is left as Yahoo returns it,
    # because a wrong GICS label is worse than an honest non-GICS one.
    YF_TO_GICS = {"Technology": "Information Technology",
                  "Communication Services": "Communication Services"}
    found = {}
    for i, t in enumerate(still, 1):
        try:
            sec_ = yf.Ticker(t).get_info().get("sector")
            if sec_:
                found[t] = YF_TO_GICS.get(sec_, sec_)
        except Exception:
            pass
        if i % 25 == 0:
            print(f"    {i}/{len(still)}, {len(found)} labelled")
        time.sleep(0.1)
    rep["sector"] = rep.sector.fillna(rep.ticker.map(found))
    print(f"  labelled {len(found)}; {int(rep.sector.isna().sum())} firm-years "
          f"remain unlabelled and are excluded from the group contrast only")

d = rep.copy()
d["idx_share"] = d.mktcap / d.groupby("year").mktcap.transform("sum")
bad = d[(d.price > PRICE_CEILING) | (d.idx_share > MAX_FIRM_SHARE)]
if len(bad):
    drop = sorted(bad.ticker.unique())
    print(f"plausibility screen drops {drop}")
    display(bad[["year", "ticker", "price", "mktcap", "idx_share", "recovered"]])
    rep = rep[~rep.ticker.isin(drop)]
else:
    print("plausibility screen: nothing flagged")

print(f"\nrepaired panel: {len(rep):,} firm-years, {rep.ticker.nunique()} firms")
print(f"  of which recovered: {int(rep.recovered.sum()):,} firm-years, "
      f"{rep[rep.recovered].ticker.nunique()} firms")

cov2 = []
for y in PAPER_YEARS:
    got = set(rep[rep.year == y].roster_ticker) | set(rep[rep.year == y].ticker)
    hit = len(ROSTER[y] & got)
    cov2.append({"year": y, "roster": len(ROSTER[y]),
                 "before": int(coverage_before.loc[coverage_before.year == y,
                                                   "in_panel"].iloc[0]),
                 "after": len(rep[rep.year == y]),
                 "coverage_after": hit / len(ROSTER[y])})
cov_cmp = pd.DataFrame(cov2)
cov_cmp["pct_before"] = (100 * coverage_before.coverage.values).round(1)
cov_cmp["pct_after"] = (100 * cov_cmp.coverage_after).round(1)
cov_cmp["gain"] = (cov_cmp.pct_after - cov_cmp.pct_before).round(1)
print("\ncoverage, before and after repair:")
display(cov_cmp[["year", "roster", "before", "after", "pct_before", "pct_after", "gain"]])
print("\nThe number to watch is `gain`. If it is large in the early years and")
print("small in the late ones, the repair has removed a composition TREND, and")
print("that trend was running through the middle of every estimate in the paper.")
savetable(rep, "repaired_panel")
savetable(cov_cmp, "coverage_before_after")

## 10 · Re-estimate with the paper's own estimator code

In [ ]:
# =============================================================================
# CELL 10 - Re-estimate, using the paper's own estimators
#
# Definitions copied from the validated repository so the two sets of numbers
# are produced by identical code. Any difference in the results is a difference
# in the DATA, which is the whole point of the exercise.
#
# Convention note: the Hill estimator targets the SURVIVAL exponent. The
# `powerlaw` package returns the DENSITY exponent and the two differ by exactly
# one; csn_fit converts, and powerlaw 2.0 additionally caps the density exponent
# at 3, so it cannot report a survival alpha above 2.
# =============================================================================
HILL_FRAC, MIN_TAIL_K, MIN_N_ALPHA, MIN_N_INEQ, GROUP_MIN_K = 0.10, 20, 50, 30, 10

def gini(x):
    x = np.sort(np.asarray(x, dtype=float)); n = len(x)
    if n == 0 or x.sum() == 0:
        return np.nan
    idx = np.arange(1, n + 1)
    return float((2 * (idx * x).sum()) / (n * x.sum()) - (n + 1) / n)

def top_share(x, q):
    x = np.sort(np.asarray(x, dtype=float))[::-1]
    if not len(x) or x.sum() == 0:
        return np.nan
    k = max(1, int(round(q * len(x))))
    return float(x[:k].sum() / x.sum())

def hhi(x):
    s = np.asarray(x, dtype=float)
    if not len(s) or s.sum() == 0:
        return np.nan
    s = s / s.sum()
    return float((s ** 2).sum())

def hill_alpha(x, k=None):
    x = np.sort(np.asarray(x, dtype=float))[::-1]; x = x[x > 0]
    if k is None:
        k = max(10, int(HILL_FRAC * len(x)))
    k = min(k, len(x) - 1)
    if k < 5:
        return np.nan, np.nan, k
    a = 1.0 / np.mean(np.log(x[:k]) - np.log(x[k]))
    return float(a), float(a / np.sqrt(k)), int(k)

def measures(panel, value="mktcap"):
    rows = []
    for y, g in panel.groupby("year"):
        x = g[value].dropna().values
        x = x[x > 0]
        if len(x) < MIN_N_INEQ:
            continue
        rec = {"year": int(y), "n": len(x), "total_cap": float(x.sum()),
               "gini": gini(x), "hhi": hhi(x),
               "top1_share": top_share(x, .01), "top5_share": top_share(x, .05),
               "top10_share": top_share(x, .10)}
        if len(x) >= MIN_N_ALPHA:
            a, se, k = hill_alpha(x, int(HILL_FRAC * len(x)))
            rec.update({"alpha_hill": a, "alpha_hill_se": se, "hill_k": k,
                        "tail_ok": k >= MIN_TAIL_K})
        rows.append(rec)
    return pd.DataFrame(rows).sort_values("year").reset_index(drop=True)

new_est = measures(rep)
old_est = pub_ineq.merge(pub_alpha[["year", "alpha_hill", "alpha_hill_se", "hill_k"]],
                         on="year", how="left")

cmp_cols = ["n", "gini", "top1_share", "top10_share", "hhi", "alpha_hill"]
comp = old_est[["year"] + [c for c in cmp_cols if c in old_est.columns]].merge(
    new_est[["year"] + cmp_cols], on="year", suffixes=("_published", "_repaired"))
for c in cmp_cols:
    if f"{c}_published" in comp.columns:
        comp[f"{c}_delta"] = comp[f"{c}_repaired"] - comp[f"{c}_published"]

print("PUBLISHED vs REPAIRED\n")
for c in ["n", "gini", "top1_share", "top10_share", "alpha_hill"]:
    sub = comp[["year", f"{c}_published", f"{c}_repaired", f"{c}_delta"]].copy()
    sub.columns = ["year", "published", "repaired", "delta"]
    print(f"--- {c} ---")
    print(sub.round(4).to_string(index=False))
    print()
savetable(comp, "published_vs_repaired")
savetable(new_est, "inequality_alpha_repaired")

## 11 · Do the published conclusions survive?

In [ ]:
# =============================================================================
# CELL 11 - Do the paper's conclusions survive?
#
# The headline claims, restated as things that can fail:
#
#   C1  concentration rose on every measure, all at p < 0.0001 except alpha
#   C2  the Hill tail index fell from 1.491 to 1.141 at p = 0.006
#   C3  the entire movement sits in the digital group
#
# The trend tests use the same specification as the paper: OLS on centred year
# with Newey-West standard errors at three lags, and a plain OLS p-value
# alongside, because ten annual observations is thin for HAC.
# =============================================================================
import statsmodels.api as sm
from scipy import stats as st

def hac_trend(df, col, label, maxlags=3):
    d = df[["year", col]].dropna().sort_values("year")
    if len(d) < 5:
        return None
    X = sm.add_constant(d.year - d.year.mean())
    fit = sm.OLS(d[col], X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    ols = sm.OLS(d[col], X).fit()
    return {"measure": col, "label": label,
            "first": float(d[col].iloc[0]), "last": float(d[col].iloc[-1]),
            "slope": float(fit.params.iloc[1]), "p_hac": float(fit.pvalues.iloc[1]),
            "p_ols": float(ols.pvalues.iloc[1]), "n_years": len(d)}

print("TREND TESTS ON THE REPAIRED PANEL\n")
rows = []
for col, lab in [("alpha_hill", "Hill alpha"), ("gini", "Gini"),
                 ("top1_share", "top 1% share"), ("top10_share", "top 10% share"),
                 ("hhi", "HHI")]:
    r_ = hac_trend(new_est, col, lab)
    if r_:
        rows.append(r_)
        print(f"  {lab:<16} {r_['first']:7.4f} -> {r_['last']:7.4f}   "
              f"slope {r_['slope']:+.4f}/yr   HAC p={r_['p_hac']:.4f}  "
              f"OLS p={r_['p_ols']:.4f}   "
              f"{'SIGNIFICANT' if r_['p_hac'] < 0.05 else 'not significant'}")
trends_new = pd.DataFrame(rows)
savetable(trends_new, "trend_tests_repaired")

PUBLISHED = {"alpha_hill": (1.491, 1.141, 0.006), "gini": (0.586, 0.719, 0.0001),
             "top1_share": (0.113, 0.300, 0.0001), "top10_share": (0.481, 0.638, 0.0001),
             "hhi": (0.0089, 0.0234, 0.0001)}
print("\n" + "=" * 72)
print("DO THE PUBLISHED CLAIMS SURVIVE?")
print("=" * 72)
verdicts = []
for _, r_ in trends_new.iterrows():
    pub = PUBLISHED.get(r_.measure)
    if not pub:
        continue
    pf, pl, pp = pub
    same_sign = np.sign(r_.last - r_.first) == np.sign(pl - pf)
    still_sig = r_.p_hac < 0.05
    shift = abs(r_.last - pl) / max(abs(pl), 1e-9)
    verdicts.append({"measure": r_.measure, "published_2016": pf, "published_2025": pl,
                     "repaired_2016": round(r_.first, 4), "repaired_2025": round(r_.last, 4),
                     "rel_shift_2025": round(shift, 4), "p_hac": round(r_.p_hac, 4),
                     "same_direction": same_sign, "still_significant": still_sig})
    flag = "HOLDS" if (same_sign and still_sig and shift < 0.10) else \
           ("DIRECTION HOLDS, MAGNITUDE MOVES" if (same_sign and still_sig) else "DOES NOT HOLD")
    print(f"\n  {r_.measure}")
    print(f"    published  {pf:.4f} -> {pl:.4f}  (p = {pp})")
    print(f"    repaired   {r_.first:.4f} -> {r_.last:.4f}  (HAC p = {r_.p_hac:.4f})")
    print(f"    2025 value moves {100*shift:.1f}%   {flag}")
vd = pd.DataFrame(verdicts)
savetable(vd, "claim_verdicts")

# --- C3, the digital contrast, with Meta present throughout -------------------
print("\n" + "=" * 72)
print("C3 -- THE DIGITAL CONTRAST, WITH THE RECOVERED FIRMS IN")
print("=" * 72)
DIGITAL_SECTORS = {"Information Technology", "Communication Services"}
DIGITAL_EXTRA = {"AMZN", "TSLA", "NFLX", "BKNG", "EBAY", "ABNB", "UBER", "DASH", "SHOP"}
sec_map = (est[["ticker", "sector"]].dropna().drop_duplicates("ticker")
           if "sector" in est.columns else pd.DataFrame(columns=["ticker", "sector"]))

g = rep.copy()   # sector was already repaired in cell 9
g["sector"] = g.sector.where(g.sector.notna(),
                             g.ticker.map(dict(zip(sec_map.ticker, sec_map.sector))))
unlab = g.sector.isna() | g.sector.astype(str).isin(["nan", "None", ""])
print(f"{int(unlab.sum())} firm-years still carry no sector label "
      f"({100*unlab.mean():.1f}% of the repaired panel)")
key_digital = ["META", "GOOGL", "GOOG", "NVDA", "AAPL", "MSFT", "AMZN"]
chk_lab = g.loc[g.ticker.isin(key_digital), ["ticker", "sector"]].drop_duplicates()
print("labels on the firms that carry the result:")
print(chk_lab.to_string(index=False))
print("  Recovered firms need labels too. Where a recovered ticker has none, it")
print("  is EXCLUDED from the contrast rather than defaulted to non-digital --")
print("  defaulting would put Meta in the wrong group and invert the result.")
g = g[~unlab]
g["group"] = np.where(g.sector.isin(DIGITAL_SECTORS)
                      | g.ticker.str.upper().isin(DIGITAL_EXTRA), "digital", "rest")

rows = []
for (y, grp), d in g.groupby(["year", "group"]):
    x = d.mktcap.dropna().values; x = x[x > 0]
    if len(x) < MIN_N_ALPHA:
        continue
    a, se, k = hill_alpha(x, max(GROUP_MIN_K, int(HILL_FRAC * len(x))))
    rows.append({"year": int(y), "group": grp, "n": len(x),
                 "alpha_hill": a, "alpha_hill_se": se, "hill_k": k})
grp_new = pd.DataFrame(rows)
if len(grp_new):
    piv_new = grp_new.pivot(index="year", columns="group", values="alpha_hill")
    piv_old = pub_group.pivot(index="year", columns="group", values="alpha_hill")
    side = piv_old.join(piv_new, lsuffix="_published", rsuffix="_repaired")
    print("\ndigital / rest tail index, published vs repaired:")
    display(side.round(4))
    print("\nfirms per group per year (repaired):")
    display(grp_new.pivot(index="year", columns="group", values="n").fillna(0).astype(int))
    if {"digital", "rest"} <= set(piv_new.columns):
        diff = (piv_new["digital"] - piv_new["rest"]).dropna()
        t, pv = st.ttest_1samp(diff, 0)
        print(f"\nmean digital-rest gap {diff.mean():+.3f}   paired t = {t:.2f}   "
              f"p = {pv:.4f}")
        print(f"published gap was -0.756 at p = 0.0001")
        print("C3:", "SUPPORTED" if diff.mean() < 0 and pv < .05 else "NOT SUPPORTED at 5%")
    savetable(grp_new, "alpha_by_group_repaired")

# Meta specifically: the case that motivated the whole repair.
meta = rep[rep.ticker == "META"].sort_values("year")
if len(meta):
    print("\nMeta in the repaired panel:")
    print(meta.assign(cap_bn=(meta.mktcap/1e9).round(0))
          [["year", "cap_bn", "recovered"]].to_string(index=False))

## 12 · Figures, manifest, download

In [ ]:
# =============================================================================
# CELL 12 - Figures, manifest, zip
# =============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
FIGS = OUTD / "figures"; FIGS.mkdir(parents=True, exist_ok=True)
INK, ACC, GREY = "#1f3b57", "#c8702a", "#7a7a7a"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "figure.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGS / f"{name}.{ext}", bbox_inches="tight")
    plt.close(fig); print(f"  saved figure {name}")

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
ax = axes[0]
ax.plot(cov_cmp.year, cov_cmp.pct_before, "o-", color=ACC, lw=2, ms=4, label="published")
ax.plot(cov_cmp.year, cov_cmp.pct_after, "o-", color=INK, lw=2, ms=4, label="repaired")
ax.set_ylabel("% of the year's roster in the panel"); ax.set_xlabel("year")
ax.set_title("Coverage"); ax.legend(fontsize=8)

for ax, col, lab in [(axes[1], "top1_share", "top 1% share of index value"),
                     (axes[2], "alpha_hill", "Hill tail index")]:
    ax.plot(comp.year, comp[f"{col}_published"], "o-", color=ACC, lw=2, ms=4,
            label="published")
    ax.plot(comp.year, comp[f"{col}_repaired"], "o-", color=INK, lw=2, ms=4,
            label="repaired")
    ax.set_xlabel("year"); ax.set_title(lab); ax.legend(fontsize=8)
fig.suptitle("The repair, and what it does to the two headline series",
             fontsize=10, color=GREY, y=1.02)
savefig(fig, "R1_published_vs_repaired")

files = sorted(p for p in OUTD.rglob("*") if p.is_file())
rows = []
for p in files:
    row = {"file": str(p.relative_to(OUTD)), "kb": round(p.stat().st_size / 1024, 1)}
    if p.suffix == ".parquet":
        try:
            row["rows"] = len(pd.read_parquet(p))
        except Exception:
            row["rows"] = None
    rows.append(row)
manifest = pd.DataFrame(rows)
manifest.to_csv(OUTD / "MANIFEST.csv", index=False)
display(manifest)

summary = {
    "run": RUN_TAG,
    "when": datetime.datetime.now().isoformat(timespec="seconds"),
    "aliases_applied": ALIASES,
    "recovered_firm_years": int(len(recovered)) if len(recovered) else 0,
    "recovered_firms": int(recovered.ticker.nunique()) if len(recovered) else 0,
    "panel_before": int(len(est)), "panel_after": int(len(rep)),
    "coverage_before": cov_cmp.pct_before.tolist(),
    "coverage_after": cov_cmp.pct_after.tolist(),
    "verdicts": vd.to_dict("records") if "vd" in dir() and len(vd) else [],
    "tables": saved_tables,
}
(OUTD / "run_summary.json").write_text(json.dumps(summary, indent=2, default=str))

print("\n" + "=" * 72)
print("WHAT TO SEND ME")
print("=" * 72)
print("  published_vs_repaired.csv   the year-by-year comparison")
print("  claim_verdicts.csv          whether each headline claim survives")
print("  coverage_before_after.csv   how much of the gap the repair closed")
print("  ticker_aliases_applied.csv  what was changed, and on what evidence")
print("\nIf the claims hold, the paper gains a robustness check and we tell the")
print("editor about it. If they move, we tell the editor before a referee does.")

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
archive = shutil.make_archive(str(BASE / f"economy_of_bits_{RUN_TAG}_{stamp}"),
                              "zip", root_dir=OUTD.parent, base_dir=OUTD.name)
print(f"\nzipped -> {archive}  ({pathlib.Path(archive).stat().st_size/1024:.0f} KB)")
try:
    from google.colab import files as colab_files
    colab_files.download(archive)
    print("Download started. Allow pop-ups if the browser blocked it.")
except ImportError:
    print("Not on Colab. Copy the archive from the path above.")